# 01 — Make Params
**Run this once ever.**  Builds the thwack template from manually-labelled hits
and saves it to `params/`.  After this, `02_detect_and_cut.ipynb` handles
everything for every new video.

Only edit the `EDITED_WAV` path below.

In [1]:
# ── Only thing to edit ───────────────────────────────────────────────────────
# Full path to the training audio (the edited video WAV with your manual labels)
EDITED_WAV = r"C:\ThatFamily Dropbox\Sorin Jayaweera\allSaves\mudd\junior\Digital Signal Processing\code\TENNIS\videosandaudio\Raja vs Wijemanne Edited.wav"
# ─────────────────────────────────────────────────────────────────────────────

In [2]:
import numpy as np
from pathlib import Path
from scipy.signal import butter, sosfiltfilt
import matplotlib.pyplot as plt
import librosa, librosa.display
import IPython.display as ipd

# Paths relative to this notebook (RunDirectory/)
HERE        = Path.cwd()
PARAMS_DIR  = HERE / "params"
LABELS_FILE = HERE / "labels" / "ballhit_timestamps_editedvideo_2.txt"
PARAMS_DIR.mkdir(exist_ok=True)

# Template settings
PRE_MS, POST_MS = 30, 80
N_FFT, HOP      = 2048, 512

print(f"Labels : {LABELS_FILE}")
print(f"Params : {PARAMS_DIR}")
%matplotlib inline

Labels : c:\ThatFamily Dropbox\Sorin Jayaweera\allSaves\mudd\junior\Digital Signal Processing\code\TENNIS\RunDirectory\thwack only processing\labels\ballhit_timestamps_editedvideo_2.txt
Params : c:\ThatFamily Dropbox\Sorin Jayaweera\allSaves\mudd\junior\Digital Signal Processing\code\TENNIS\RunDirectory\thwack only processing\params


In [3]:
def load_labels(path):
    rows = []
    with open(path) as f:
        for line in f:
            p = line.strip().split()
            if len(p) >= 2:
                rows.append([float(p[0]), float(p[1])])
    return np.array(rows)

def bandpass(audio, sr, lo=1000.0, hi=8000.0, order=4):
    nyq = 0.5 * sr
    sos = butter(order, [lo/nyq, hi/nyq], btype="bandpass", output="sos")
    return sosfiltfilt(sos, audio).astype(np.float32)

def build_template(audio, sr, labels, pre_ms=PRE_MS, post_ms=POST_MS,
                   n_fft=N_FFT, hop=HOP):
    pre_s, post_s = int(pre_ms/1000*sr), int(post_ms/1000*sr)
    mags, maxf = [], 0
    for t0, t1 in labels:
        seg = audio[int(t0*sr):int(t1*sr)]
        if not len(seg): continue
        c = int(t0*sr) + np.argmax(np.abs(seg))
        ws, we = c - pre_s, c + post_s
        if ws < 0 or we > len(audio): continue
        a = audio[ws:we].astype(np.float32)
        a /= np.linalg.norm(a) + 1e-8
        m = np.abs(librosa.stft(a, n_fft=n_fft, hop_length=hop))
        mags.append(m); maxf = max(maxf, m.shape[1])
    padded = [np.pad(m, ((0,0),(0,maxf-m.shape[1]))) for m in mags]
    return np.mean(np.stack(padded), axis=0).astype(np.float32)

## Build template

In [4]:
labels    = load_labels(LABELS_FILE)
audio_raw, sr = librosa.load(str(EDITED_WAV), sr=None)
audio_bp  = bandpass(audio_raw, sr)
print(f"Audio: {len(audio_raw)/sr/60:.1f} min @ {sr} Hz  |  Labels: {len(labels)}")

template = build_template(audio_bp, sr, labels)
print(f"Template: {template.shape}")

freqs = librosa.fft_frequencies(sr=sr, n_fft=N_FFT)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
img = librosa.display.specshow(librosa.amplitude_to_db(template, ref=np.max),
    sr=sr, hop_length=HOP, x_axis="time", y_axis="log", cmap="magma", ax=axes[0])
axes[0].set_title("Template spectrogram"); fig.colorbar(img, ax=axes[0], format="%+2.0f dB")
p = template.mean(axis=1)
axes[1].semilogx(freqs, 20*np.log10(p/p.max()+1e-9))
axes[1].axvspan(1000,8000,alpha=0.1,color="green",label="bandpass 1-8 kHz")
axes[1].set_xlim(100,sr/2); axes[1].set_ylim(-60,5)
axes[1].set_xlabel("Hz"); axes[1].set_ylabel("dB"); axes[1].legend(); axes[1].grid(alpha=0.3,which="both")
plt.tight_layout(); plt.show()

FileNotFoundError: [Errno 2] No such file or directory: 'c:\\ThatFamily Dropbox\\Sorin Jayaweera\\allSaves\\mudd\\junior\\Digital Signal Processing\\code\\TENNIS\\RunDirectory\\thwack only processing\\labels\\ballhit_timestamps_editedvideo_2.txt'

## Listen to training hits

In [ ]:
pre_s, post_s = int(30/1000*sr), int(300/1000*sr)
clips = []
for t0, t1 in labels:
    c = int(t0*sr) + np.argmax(np.abs(audio_raw[int(t0*sr):int(t1*sr)]))
    ws, we = max(0,c-pre_s), min(len(audio_raw),c+post_s)
    snip = audio_raw[ws:we].astype(np.float32); snip /= np.max(np.abs(snip))+1e-8
    full = np.zeros(pre_s+post_s, dtype=np.float32)
    full[pre_s-(c-ws):pre_s-(c-ws)+len(snip)] = snip
    clips.append(full)
avg = np.mean(np.stack(clips),axis=0); avg /= np.max(np.abs(avg))+1e-8
print(f"Average of {len(clips)} training hits:")
ipd.display(ipd.Audio(avg, rate=sr))

Average of 57 training hits:


## Save — done forever

In [ ]:
np.save(PARAMS_DIR/"template.npy",   template)
np.save(PARAMS_DIR/"sr.npy",         np.array([sr]))
np.save(PARAMS_DIR/"hop_length.npy", np.array([HOP]))
np.save(PARAMS_DIR/"n_fft.npy",      np.array([N_FFT]))
np.save(PARAMS_DIR/"pre_ms.npy",     np.array([PRE_MS]))

print("Saved to params/:")
for f in sorted(PARAMS_DIR.iterdir()):
    print(f"  {f.name:30s}  {f.stat().st_size/1024:.1f} KB")
print("\nDone. Run 02_detect_and_cut.ipynb for every new video.")

Saved to params/:
  hop_length.npy                  0.1 KB
  n_fft.npy                       0.1 KB
  pre_ms.npy                      0.1 KB
  sr.npy                          0.1 KB
  template.npy                    40.2 KB

Done. Run 02_detect_and_cut.ipynb for every new video.
